# Model comparison table

`build_model_comparison_table(run_id)` builds one row per model under `results/<run_id>/` with:

- Structure adherence (`valid_structure_count/bt_count`)
- Structure adherence % (2 dp)
- Task completion % (2 dp)
- Avg inference time (seconds)

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT_DIR = Path.cwd().resolve().parents[2]
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.utils import get_results_dir


def build_model_comparison_table(run_id: str) -> pd.DataFrame:
    results_dir = get_results_dir(run_id)
    rows = []
    for model_dir in sorted(p for p in results_dir.iterdir() if p.is_dir()):
        main_path = model_dir / "main_results.json"
        task_path = model_dir / "tasks" / "task_go_to_target_v1.json"
        if not main_path.exists() or not task_path.exists():
            continue

        main = json.loads(main_path.read_text(encoding="utf-8"))
        all_tasks = main.get("all_tasks", [])
        if not all_tasks:
            continue

        task_summary = all_tasks[0]
        valid_structure_count = int(task_summary.get("valid_structure_count", 0))
        bt_count = int(task_summary.get("bt_count", 0))
        adherence_ratio = f"{valid_structure_count}/{bt_count}" if bt_count else "0/0"

        rows.append({
            "model_id": main.get("model_id", model_dir.name),
            "Structure adherence": adherence_ratio,
            "Structure adherence %": f"{float(main.get('structure_adherence_pct', 0.0)):.2f}%",
            "Task completion %": f"{float(main.get('task_completion_pct', 0.0)):.2f}%",
            "avg_inference_time": round(float(main.get("avg_inference_time_s", 0.0)), 3),
        })

    return pd.DataFrame(rows).sort_values("model_id").reset_index(drop=True)

In [ ]:
build_model_comparison_table("bt_online_one_task")